In [1]:
!pip install yfinance fredapi

In [2]:
import yfinance as yf
import pandas as pd
from fredapi import Fred

# ── FRED API key ──────────────────────────────────────────────
FRED_API_KEY = "f88af3abe0ec02cc1f660faa83b41d27"
fred = Fred(api_key=FRED_API_KEY)

# ── Date range ────────────────────────────────────────────────
START = "2017-01-01"
END    = "2026-03-31"

# ── Pull from Yahoo Finance ───────────────────────────────────
btc = yf.download("BTC-USD", start=START, end=END)["Close"]
btc.name = "BTC"
spx = yf.download("^GSPC",   start=START, end=END)["Close"]
spx.name = "SPX"
vix = yf.download("^VIX",    start=START, end=END)["Close"]
vix.name = "VIX"
dxy = yf.download("DX-Y.NYB",start=START, end=END)["Close"]
dxy.name = "DXY"

# ── Pull from FRED ────────────────────────────────────────────
treasury = fred.get_series("DGS10",  observation_start=START, observation_end=END)
treasury.name = "US10Y"
breakeven = fred.get_series("T10YIE", observation_start=START, observation_end=END)
breakeven.name = "BREAKEVEN"

# ── Merge everything ──────────────────────────────────────────
df = pd.concat([btc, spx, vix, dxy, treasury, breakeven], axis=1)

# Keep only weekdays (FRED series have no weekends, Yahoo does)
df = df[df.index.dayofweek < 5]

# Forward fill small gaps (e.g. US holidays where FRED has no value)
df = df.ffill()

# Drop any remaining rows with missing values
df = df.dropna()

print(f"Dataset ready: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
df.head()

/tmp/ipykernel_10174/229359530.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  btc = yf.download("BTC-USD", start=START, end=END)["Close"]
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_10174/229359530.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spx = yf.download("^GSPC",   start=START, end=END)["Close"]
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_10174/229359530.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix = yf.download("^VIX",    start=START, end=END)["Close"]
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_10174/229359530.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dxy = yf.download("DX-Y.NYB",start=START, end=END)["Close"]
[*********************100%***********************]  1 of 1 completed


Dataset ready: 2411 rows, 6 columns
Date range: 2017-01-03 to 2026-03-31


,BTC-USD,^GSPC,^VIX,DX-Y.NYB,US10Y,BREAKEVEN
2017-01-03,1043.839966,2257.830078,12.85,103.209999,2.45,1.98
2017-01-04,1154.729980,2270.750000,11.85,102.699997,2.46,1.99
2017-01-05,1013.380005,2269.000000,11.67,101.519997,2.37,1.95
2017-01-06,902.200989,2276.979980,11.32,102.220001,2.42,1.96
2017-01-09,902.828003,2268.899902,11.56,101.930000,2.38,1.95


In [3]:
# ── Convert prices to daily returns ──────────────────────────
# BTC and SPX: we want returns, not raw prices
# VIX, DXY, US10Y, BREAKEVEN: keep as levels (they already represent rates/indices)

df["BTC_ret"] = df["BTC-USD"].pct_change()
df["SPX_ret"] = df["^GSPC"].pct_change()

# Drop the first row (NaN from pct_change)
df = df.dropna()

# Save the clean file
df.to_csv("data_clean.csv")
print("Saved as data_clean.csv")
print(df.describe().round(4))

Saved as data_clean.csv
           BTC-USD      ^GSPC       ^VIX   DX-Y.NYB      US10Y  BREAKEVEN  \
count    2410.0000  2410.0000  2410.0000  2410.0000  2410.0000  2410.0000   
mean    34764.7316  4035.7599    18.8027    98.6941     2.8304     2.1107   
std     32530.2016  1296.4777     7.4570     5.1349     1.2022     0.3498   
min       777.7570  2237.3999     9.1400    88.5900     0.5200     0.5000   
25%      8186.2025  2857.2125    13.7400    94.7500     1.8000     1.8700   
50%     23276.8027  3939.9651    17.0900    98.1200     2.8300     2.2100   
75%     56369.7090  4724.2300    21.7700   102.7675     4.0500     2.3400   
max    124752.5312  6978.6001    82.6900   114.1100     4.9800     3.0200   

         BTC_ret    SPX_ret  
count  2410.0000  2410.0000  
mean      0.0026     0.0005  
std       0.0422     0.0114  
min      -0.3717    -0.1198  
25%      -0.0160    -0.0035  
50%       0.0014     0.0005  
75%       0.0211     0.0057  
max       0.2525     0.0952  


In [4]:
from google.colab import files
files.download("data_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>